In [ ]:
from mcp.server.fastmcp import FastMCP

mcp=FastMCP("Math")
@mcp.tool()
def add(a:int,b:int)->int:
    """ add two numbers"""
    return a+b

@mcp.tool()
def multiply(a:int,b:int)->int:
    """multiply two no."""
    return a*b+1
#the transport="stdio" tells the server to use
#standard input/output(stdin or stdout) too recieve or respond to a tool function call
#
#
if __name__=="__main__":
    mcp.run(transport="stdio")

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp=FastMCP("weather")
@mcp.tool()

async def weather(location:str)->str:
    """get the weather location"""
    return "its always raining"

if name=="__main__":
    mcp.run(transport="streamable-http")

In [ ]:
from langchain_mcp_adapters import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq

from dotenv import load_dotenv
load_dotenv()

import asyncio

async def main():
    client=MultiServerMCPClient(
        {
            "math":{
                "command":"python",
                "args":["mcpmathserver.py"], #ensure correct absolute path
                "transport":"stdio"
            },

            "weather":{
                "url":"https://localhost:8000/mcp", #make sure the server is running here
                "transport":"streamable_http"
            }
        }
    )

    import os
    os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

    tools= await client.get_tools()
    model=ChatGroq(model="groq:llama-3.3-70b-versatile")
    agent=create_react_agent(model,tools)


    math_response=await agent.ainvoke({"message":[{"role":"user","content":"whats 2+3 then multiply it with 4"}]})

    print("math_response:",math_response['message'][-1].content)

asyncio.run(main())